# Day 10 · Exercise 2: Build a Template

**What you'll build:** `render_template(template_str: str, variables: dict) -> str` — a function that wraps `string.Template` to fill a prompt template string using a variables dictionary, using `substitute()` when all keys are present and exposing `safe_substitute()` behaviour through an optional flag.

**Why it matters:** Knowing how to construct reusable, dollar-sign-based prompt templates is the foundation of a maintainable prompt library — it keeps JSON examples and code blocks inside your prompts without any brace-doubling.

## Your Implementation

In [ ]:
from string import Template


def render_template(template_str: str, variables: dict) -> str:
    """Fill a string.Template prompt using a variables dictionary.

    Creates a Template from `template_str` (which uses $variable or
    ${variable} placeholders) and calls substitute() with `variables`.
    Raises KeyError immediately if any placeholder has no matching key
    in `variables` — fail loudly so missing keys are never silently
    forwarded to the model.

    Args:
        template_str: A string containing $name or ${name} placeholders.
        variables: A dict mapping placeholder names to replacement values.

    Returns:
        The fully substituted string with all placeholders replaced.

    Example:
        >>> render_template("Hello, $name!", {"name": "world"})
        'Hello, world!'
        >>> render_template("${lang}Script is fun.", {"lang": "Java"})
        'JavaScript is fun.'
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and is callable
    try:
        assert callable(render_template), 'render_template is not defined or not callable'
        print(f'{_PASS} Check 1/{total}: render_template is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # cannot safely run remaining checks

    # Check 2: basic $variable substitution returns correct string
    try:
        result = render_template("Hello, $name!", {"name": "world"})
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        assert result == "Hello, world!", f'expected "Hello, world!", got {result!r}'
        print(f'{_PASS} Check 2/{total}: $variable placeholder substituted correctly')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: braced ${variable} form works and adjoins surrounding text
    try:
        result = render_template("${lang}Script is fun.", {"lang": "Java"})
        assert result == "JavaScript is fun.", f'expected "JavaScript is fun.", got {result!r}'
        print(f'{_PASS} Check 3/{total}: ${"{"}lang{"}"}Script braced form substituted correctly')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: substitute() raises KeyError on a missing key
    try:
        raised = False
        try:
            render_template("Topic: $topic, Limit: $limit", {"topic": "AI"})
        except KeyError:
            raised = True
        assert raised, 'render_template did not raise KeyError for missing key "limit"'
        print(f'{_PASS} Check 4/{total}: KeyError raised for missing placeholder key')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: multiple variables all substituted, no $ tokens remain
    try:
        tmpl = "Summarise $topic in $max_sentences sentences for a $audience audience."
        result = render_template(tmpl, {
            "topic": "climate change",
            "max_sentences": "3",
            "audience": "general",
        })
        assert '$' not in result, f'unresolved placeholder still present in: {result!r}'
        assert "climate change" in result
        assert "3" in result
        assert "general" in result
        print(f'{_PASS} Check 5/{total}: all three placeholders substituted, no $ tokens remain')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

In Lesson 3 you will build a full prompt library module where templates are
defined at module level and exposed through narrow builder functions. Foreshadow
that pattern now:

1. Define a module-level `SUMMARISE_TMPL = Template("...")` with three placeholders:
   `$topic`, `$max_sentences`, and `$audience`.
2. Write a `build_summarise_prompt(topic, max_sentences=3, audience="general") -> str`
   function that calls `render_template` internally.
3. Verify that calling `build_summarise_prompt("quantum computing")` produces a
   complete prompt string with no dollar-sign tokens.

This two-layer approach — template constant + builder function — is exactly the
shape a real prompt library uses.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
from string import Template


def render_template(template_str: str, variables: dict) -> str:
    """Fill a string.Template prompt using a variables dictionary.

    Args:
        template_str: A string containing $name or ${name} placeholders.
        variables: A dict mapping placeholder names to replacement values.

    Returns:
        The fully substituted string with all placeholders replaced.

    Example:
        >>> render_template("Hello, $name!", {"name": "world"})
        'Hello, world!'
    """
    tmpl = Template(template_str)
    return tmpl.substitute(variables)
```

**Why this works:** `Template(template_str)` parses the dollar-sign placeholders once and stores them; calling `.substitute(variables)` then replaces every `$name` or `${name}` token with the matching value from the dict, raising `KeyError` immediately if any placeholder is missing. This single-line body is intentionally minimal — the real design choice is *using* `substitute` rather than `safe_substitute`, so that a forgotten variable produces a loud failure rather than a silent `$variable` token being forwarded to the model.
</details>